<a href="https://colab.research.google.com/github/Prijitha7/-my-javascript-basic-projects/blob/master/groundwater_quality_2023.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import pandas as pd
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

raw = pd.read_excel(file_name, header=None)

header_row = raw[raw.apply(lambda r: r.astype(str).str.contains("pH", case=False).any(), axis=1)].index[0]

df = pd.read_excel(file_name, header=header_row)

df.columns = df.columns.astype(str).str.strip().str.lower()

df = df.loc[:, ~df.columns.str.contains("unnamed", na=False)]

numeric_cols = df.columns.difference([
    "state","district","location","latitude","longitude","water source","water type"
])


df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

df = df.dropna(subset=numeric_cols, how="all")

df.to_csv("ground_water_quality_report2023_new.csv", index=False)


Saving ground  water quality report 2023.xlsx to ground  water quality report 2023.xlsx


In [ ]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')
import os

folder_path = '/content/drive/MyDrive/Demo water'
files = os.listdir(folder_path)
print("Found files:", files)

for file in files:
    full_path = os.path.join(folder_path, file)

    if file.endswith('.xlsx') or file.endswith('.xls'):
        print(f"\nReading file: {file}")
        try:
            df = pd.read_excel(full_path)
            print(df.head())
        except Exception as e:
            print(f"Error reading {file}: {e}")
    elif file.endswith('.csv'):
        print(f"\nReading CSV file: {file}")
        try:
            df = pd.read_csv(full_path, encoding='utf-8')
        except UnicodeDecodeError:
            df = pd.read_csv(full_path, encoding='latin1')
        print(df.head())





Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found files: ['ground water quality report 2019', 'ground water quality report 2020', 'ground water quality 2021', 'ground water quality report 2022', 'ground water quality 2023']


In [1]:
import pandas as pd
import os

# Mount Google Drive (if using Colab)
from google.colab import drive
drive.mount('/content/drive')

# Folder path
folder_path = '/content/drive/MyDrive/Demo water'
files = os.listdir(folder_path)
print("Found files:", files)

# Standard column mapping
column_mapping = {
    'ph': ['ph', 'pH'],
    'ec_µs/cm_at': ['ec', 'EC', 'ec_(µs/cm_at', 'ec_(μs/cm', 'ec_in_μs/cm'],
    'co3_mg/l': ['co3', 'CO3', 'co3_mg/l', 'co3_(mg/l)'],
    'hco3': ['hco3', 'HCO3', 'bicarbonate'],
    'cl_mg/l': ['cl', 'Cl', 'cl_mg/l', 'cl_(mg/l)'],
    'so4': ['so4', 'SO4', 'sulphate'],
    'no3': ['no3', 'NO3', 'nitrate'],
    'po4': ['po4', 'PO4', 'phosphate'],
    'total_hardness': ['total_hardness', 'total hardness', 'th'],
    'ca_mg/l': ['ca', 'Ca', 'ca_mg/l', 'ca_(mg/l)'],
    'mg_mg/l': ['mg', 'Mg', 'mg_mg/l', 'mg_(mg/l)'],
    'na_mg/l': ['na', 'Na', 'na_mg/l', 'na_(mg/l)'],
    'k_mg/l': ['k', 'K', 'k_mg/l', 'k_(mg/l)'],
    'f_mg/l': ['f', 'F', 'f_mg/l', 'f_(mg/l)'],
    'fe_ppm': ['fe', 'Fe', 'fe_ppm'],
    'as_ppb': ['as', 'As', 'as_ppb'],
    'u_ppb': ['u_ppb_', 'u_ppb', 'U_ppb', 'U_(ppb)', 'U(ppb)', 'u_(ppb)']
}

all_dfs = []

for f in files:
    full_path = os.path.join(folder_path, f)

    try:
        df = pd.read_excel(full_path, engine='openpyxl')

        # Clean column names
        df.columns = (
            df.columns.astype(str)
            .str.strip()
            .str.lower()
            .str.replace(" ", "_")
            .str.replace("(", "", regex=False)
            .str.replace(")", "", regex=False)
        )

        # Rename columns using mapping
        for standard_name, variations in column_mapping.items():
            for col in df.columns:
                if col.lower() in [v.lower() for v in variations]:
                    df.rename(columns={col: standard_name}, inplace=True)

        # Keep only needed columns
        required_cols = ['state', 'year'] + list(column_mapping.keys())
        df = df[[c for c in required_cols if c in df.columns]]

        # Convert numeric columns
        numeric_cols = df.columns.difference(['state', 'year'])
        for col in numeric_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')

        # Forward fill state
        if 'state' in df.columns:
            df['state'] = df['state'].ffill()

        # Drop rows where all numeric values are NaN
        df = df.dropna(subset=numeric_cols, how='all')

        all_dfs.append(df)
        print(f"Read {f} successfully, shape: {df.shape}")

    except Exception as e:
        print(f"Could not read {f}: {e}")

# Combine and Aggregate
if all_dfs:
    combined_df = pd.concat(all_dfs, ignore_index=True)

    agg_df = combined_df.groupby(['state', 'year'], as_index=False).mean(numeric_only=True)

    # Column order
    columns_order = ['state', 'year'] + list(column_mapping.keys())
    agg_df = agg_df[[c for c in columns_order if c in agg_df.columns]]

    # Save CSV
    output_file = "groundwater_quality_2019-2023.csv"
agg_df.to_csv(output_file, index=False)

print(f"\n✅ Combined and cleaned CSV saved as '{output_file}'")

# Download file
from google.colab import files
files.download(output_file)

Mounted at /content/drive
Found files: ['ground water quality report 2022.xlsx', 'ground water quality 2021.xlsx', 'ground water quality report 2020.xlsx', 'ground water quality report 2019.xlsx', 'ground water quality 2023.xlsx']
Read ground water quality report 2022.xlsx successfully, shape: (13600, 17)
Read ground water quality 2021.xlsx successfully, shape: (5375, 16)
Read ground water quality report 2020.xlsx successfully, shape: (6970, 16)
Read ground water quality report 2019.xlsx successfully, shape: (13549, 16)
Read ground water quality 2023.xlsx successfully, shape: (16776, 18)

✅ Combined and cleaned CSV saved as 'groundwater_quality_2019-2023.csv'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>